In [9]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import optuna

df=pd.read_csv('/Users/lambardaar/Downloads/ML_final_project/Bank_Churn.csv')
df.head()

,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [10]:
df.drop(['CustomerId', 'Surname'], axis=1, inplace=True)

X = df.drop('Exited', axis=1)
y = df['Exited']


#preprocessing
cat_cols = X.select_dtypes(include=['object']).columns
num_cols = X.select_dtypes(exclude=['object']).columns

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols),
        ("num", "passthrough", num_cols)#Keep unspecified columns unchanged
    ]
)


In [11]:
#split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [12]:
def xgb_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 400),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "scale_pos_weight": (y_train == 0).sum() / (y_train == 1).sum(),
        "eval_metric": "logloss",
        "tree_method": "hist"
    }

    model = XGBClassifier(**params)

    pipe = Pipeline([
        ("prep", preprocessor),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)
    preds = pipe.predict_proba(X_test)[:, 1]

    return average_precision_score(y_test, preds)  # PR-AUC (correct for imbalance)

study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(xgb_objective, n_trials=15)

[I 2026-03-23 16:56:28,962] A new study created in memory with name: no-name-2b42dfb0-94a6-46d7-9f0b-994caa1e396e
[I 2026-03-23 16:56:29,291] Trial 0 finished with value: 0.6537025499746942 and parameters: {'n_estimators': 137, 'max_depth': 6, 'learning_rate': 0.18964512349252474, 'subsample': 0.7611866921647332, 'colsample_bytree': 0.8889113774492233}. Best is trial 0 with value: 0.6537025499746942.
[I 2026-03-23 16:56:29,560] Trial 1 finished with value: 0.6653833757159113 and parameters: {'n_estimators': 276, 'max_depth': 6, 'learning_rate': 0.10943789554731977, 'subsample': 0.9333154560341469, 'colsample_bytree': 0.847419375974859}. Best is trial 1 with value: 0.6653833757159113.
[I 2026-03-23 16:56:29,888] Trial 2 finished with value: 0.6671192396891887 and parameters: {'n_estimators': 217, 'max_depth': 7, 'learning_rate': 0.11682540418338787, 'subsample': 0.7787894532204807, 'colsample_bytree': 0.7622336477488341}. Best is trial 2 with value: 0.6671192396891887.
[I 2026-03-23 16:

In [ ]:
def lgb_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 400),
        "max_depth": trial.suggest_int("max_depth", -1, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
        "num_leaves": trial.suggest_int("num_leaves", 20, 120),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "class_weight": "balanced"
    }

    model = LGBMClassifier(**params)

    pipe = Pipeline([
        ("prep", preprocessor),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)
    preds = pipe.predict_proba(X_test)[:, 1]

    return average_precision_score(y_test, preds)

study_lgb = optuna.create_study(direction="maximize")
study_lgb.optimize(lgb_objective, n_trials=15)


EVALUATING
BEST MODEL

In [14]:
best_xgb = XGBClassifier(**study_xgb.best_params)
best_lgb = LGBMClassifier(**study_lgb.best_params)

pipe_xgb = Pipeline([("prep", preprocessor), ("model", best_xgb)])
pipe_lgb = Pipeline([("prep", preprocessor), ("model", best_lgb)])

pipe_xgb.fit(X_train, y_train)
pipe_lgb.fit(X_train, y_train)


[LightGBM] [Info] Number of positive: 1630, number of negative: 6370
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000158 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 857
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.203750 -> initscore=-1.363019
[LightGBM] [Info] Start training from score -1.363019
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  Index(['Geography', 'Gender'], dtype='object')),
                                                 ('num', 'passthrough',
                                                  Index(['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary'],
      dtype='object'))])),
                ('model',
                 LGBMClassifier(colsample_bytree=0.927620902061019,
                                learning_rate=0.06492453433635544, max_depth=5,
                                n_estimators=182, num_leaves=40,
                                subsample=0.8076283074470363))])

In [15]:
xgb_pred = pipe_xgb.predict_proba(X_test)[:, 1]
lgb_pred = pipe_lgb.predict_proba(X_test)[:, 1]

xgb_roc = roc_auc_score(y_test, xgb_pred)
xgb_pr  = average_precision_score(y_test, xgb_pred)

lgb_roc = roc_auc_score(y_test, lgb_pred)
lgb_pr  = average_precision_score(y_test, lgb_pred)

print("XGBoost -> ROC:", xgb_roc, "PR:", xgb_pr)
print("LightGBM -> ROC:", lgb_roc, "PR:", lgb_pr)

XGBoost -> ROC: 0.8663108408871122 PR: 0.7135977357881746
LightGBM -> ROC: 0.8644191186564067 PR: 0.7109122508316748


In [16]:
if xgb_pr >= lgb_pr:
    final_model = pipe_xgb
    chosen = "XGBoost"
else:
    final_model = pipe_lgb
    chosen = "LightGBM"

print("Selected Model:", chosen)

Selected Model: XGBoost


In [19]:
import joblib
joblib.dump(final_model, "churn_model.pkl")

['churn_model.pkl']